# Ablation run — single-pass baseline vs full pipeline (spec §8 / M5)

Purpose-built for one job, so it can be run top to bottom without touching the
VLM sections in `flowmind_colab.ipynb`.

**Before running:** attach a **GPU** runtime. In VS Code that's
`Select Kernel → Colab → GPU`; in the browser, `Runtime → Change runtime type`.

**Runs on `p/examiner`, not `main`** — the Examiner only exists on that branch.

Budget about an hour. Roughly 30 GB of model weights come down (Qwen3-8B for
answering, Mistral-7B for judging) plus generation time. Model weights go to the
VM disk, not Drive: Qwen3-8B alone is 16 GB and free Drive is 15 GB total. That
means a recycled runtime re-downloads them, which is the accepted trade.

Results are written to `runs/` which is symlinked to Drive, so output survives a
disconnect even if the weights don't.

## 0. Confirm a GPU is attached

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || print('NO GPU — change the runtime type before continuing')

## 1. Setup

Idempotent — safe to re-run on a fresh runtime. Expect the log line to show
branch `p/examiner` and **52 passed**. If it shows `main`, stop: the Examiner
isn't there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/FlowMind')
REPO = Path('/content/FlowMind')

if not REPO.exists():
    !git clone -q https://github.com/PurvajaNarayan/FlowMind.git {REPO}
!cd {REPO} && git fetch -q origin && git checkout -q p/examiner && git pull -q --ff-only
!cd {REPO} && git log --oneline -1 && git branch --show-current

# Point the gitignored paths at Drive. `runs` matters most here: it is where the
# generated answers land, and re-generating them is the expensive part.
for rel in ('data/train_full.json', 'data/test_full.json', 'data/images',
            'models', 'runs'):
    link, target = REPO / rel, DRIVE / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        link.unlink()
    elif link.is_dir():
        shutil.rmtree(link)
    elif link.exists():
        link.unlink()
    link.symlink_to(target)
    print(f'{rel:24} -> {target}')

# bitsandbytes is NOT in requirements-vlm.txt (commented out there). Without it,
# 4-bit loading fails and Qwen3-8B in fp16 (~16GB) will not fit a 15GB T4.
!cd {REPO} && pip install -q -r requirements.txt && pip install -q bitsandbytes accelerate
!cd {REPO} && python -m pytest -q 2>&1 | tail -2

## 2. Smoke test — 12 items

Most of the time here is the 16 GB Qwen3-8B download, not generation. Three
things to check in the output:

- **no `[llm] 4-bit unavailable`** — if that appears, bitsandbytes didn't take and
  the model won't fit
- topological rows show `OK`/`X`; content rows show `--`, meaning recorded but not
  scored inline (correct — the judge does those in step 4)
- `examiner fired a revision : N` — any value; it just proves the loop runs

In [ ]:
!cd /content/FlowMind && HF_HOME=/content/hf_llm python tools/run_ablation.py --n 12 --save runs/abl_smoke.jsonl

## 3. Full run — 60 items

Weights are cached from step 2, so this is generation only. 60 items spread over
the 12 (subset × question-type) cells, max 2 questions per flowchart.

In [ ]:
!cd /content/FlowMind && HF_HOME=/content/hf_llm python tools/run_ablation.py --n 60 --save runs/ablation_v1.jsonl

## 4. Score both arms with the judge

Downloads Mistral-7B (~15 GB) once. Read the output in this order:

1. **`judge agrees with exact match: N/…`** — the judge measured against known
   truth on the topological questions. If this is low, treat everything below as
   correspondingly noisy.
2. **content accuracy by arm, and the delta** — the project's headline, and the
   first real number for the 57% of the benchmark that needs language.
3. **`unparsed judge replies`** — these are excluded from both numerator and
   denominator, so a large count means the judge is struggling with the format
   rather than that answers were wrong.

In [ ]:
!cd /content/FlowMind && HF_HOME=/content/hf_llm python tools/score_run.py runs/ablation_v1.jsonl --save runs/ablation_v1_scored.jsonl

### If step 4 fails on a gated repo

`mistralai/*` repos sometimes require accepting terms. Either accept on the HF
model page and set `HF_TOKEN`, or fall back to Phi-4-mini, which is MIT and
ungated — the cell below does that. Note in the write-up which judge produced the
numbers.

In [ ]:
# only needed if the cell above failed to download the judge
!cd /content/FlowMind && HF_HOME=/content/hf_llm FLOWMIND_JUDGE_MODEL=microsoft/Phi-4-mini-instruct python tools/score_run.py runs/ablation_v1.jsonl --save runs/ablation_v1_scored.jsonl